# Model Training Pipeline (Granular Dual-Horizons)
Trains explicitly parallel models for Both 3 and 12 Hour warnings.


### 1. Library Imports


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


### 2. Load Processed Data


In [ ]:
df = pd.read_csv('processed_training_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])


### 3. Explicit Feature Inclusion Array
Rigidly defines exactly what numeric and encoded arrays enter the model, implicitly discarding unencoded raw strings.


In [ ]:
features = [
    'water_level', 'rainfall_mm', 'minor_flood_level', 'major_flood_level',
    'gap_flag_6h', 'gap_flag_24h', 'water_level_was_missing', 'rainfall_was_missing',
    'hour', 'day', 'month', 'day_of_week', 'quarter', 'is_monsoon',
    'water_level_delta', 
    'water_level_lag_1', 'water_level_lag_2', 'water_level_lag_4', 'water_level_lag_8', 'water_level_lag_16', 'water_level_lag_24',
    'rainfall_lag_1', 'rainfall_lag_2', 'rainfall_lag_4', 'rainfall_lag_8',
    'water_level_roll_mean_3', 'water_level_roll_max_3',
    'rainfall_roll_sum_3', 'rainfall_roll_sum_6', 'rainfall_roll_sum_8', 'rainfall_roll_sum_12', 'rainfall_roll_sum_16',
    'flood_ratio_minor', 'flood_ratio_major', 'above_minor_flag', 'above_major_flag',
    'station_encoded', 'river_basin_encoded', 'rainfall_type_encoded', 'status_encoded'
]
features = [f for f in features if f in df.columns]


### 4. Mathematical Data Routing (Chronological Split)


In [ ]:
df_train = df[df['datetime'] < '2025-01-01']
df_test = df[df['datetime'] >= '2025-01-01']

X_train = df_train[features].replace([np.inf, -np.inf], np.nan)
X_test = df_test[features].replace([np.inf, -np.inf], np.nan)


### 5. Train Model (3-Hour Future Horizon)


In [ ]:
y_train_3h = df_train['target_water_level_3h']

model_3h = xgb.XGBRegressor(
    n_estimators=250, 
    learning_rate=0.03, 
    max_depth=7, 
    subsample=0.75, 
    colsample_bytree=0.6, 
    random_state=42, 
    n_jobs=-1
)
model_3h.fit(X_train, y_train_3h)
print("3-Hour Horizon model trained!")


### 6. Train Model (12-Hour Future Horizon)


In [ ]:
y_train_12h = df_train['target_water_level_12h']

model_12h = xgb.XGBRegressor(
    n_estimators=250, 
    learning_rate=0.03, 
    max_depth=7, 
    subsample=0.75, 
    colsample_bytree=0.6, 
    random_state=42, 
    n_jobs=-1
)
model_12h.fit(X_train, y_train_12h)
print("12-Hour Deep Horizon model trained!")


### 7. Evaluation (3-Hour Model)
Scores MAE strictly on timestamps directly measured by a human.


In [ ]:
y_pred_3h = model_3h.predict(X_test)
df_test_3h = df_test.copy()
df_test_3h['pred'] = y_pred_3h
df_test_3h['error_abs'] = np.abs(df_test_3h['target_water_level_3h'] - df_test_3h['pred'])

physical_3h = df_test_3h[df_test_3h['water_level_was_missing'] == 0]
print(f"3-Hour Forecast MAE (Physical Native Readings Only): {physical_3h['error_abs'].mean():.4f}")


### 8. Evaluation (12-Hour Model)
Scores 12-hour predictive capability strictly on timestamps directly measured.


In [ ]:
y_pred_12h = model_12h.predict(X_test)
df_test_12h = df_test.copy()
df_test_12h['pred'] = y_pred_12h
df_test_12h['error_abs'] = np.abs(df_test_12h['target_water_level_12h'] - df_test_12h['pred'])

physical_12h = df_test_12h[df_test_12h['water_level_was_missing'] == 0]
print(f"12-Hour Forecast MAE (Physical Native Readings Only): {physical_12h['error_abs'].mean():.4f}")

print("----------")
print("Top 10 Worst Errors evaluating 12-Hour Forecast (Physical readings only):")
print(physical_12h.groupby('station')['error_abs'].mean().sort_values(ascending=False).head(10))


### 9. Visualization: Feature Importance
Let's visually compare what physics signals drive short-term vs long-term forecasts.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 3-Hour Importance
imp_3h = pd.DataFrame({'Feature': features, 'Importance': model_3h.feature_importances_}).sort_values('Importance', ascending=False)
sns.barplot(data=imp_3h.head(15), x='Importance', y='Feature', ax=axes[0])
axes[0].set_title('Top 15 Features (3-Hour Forecast)')

# 12-Hour Importance
imp_12h = pd.DataFrame({'Feature': features, 'Importance': model_12h.feature_importances_}).sort_values('Importance', ascending=False)
sns.barplot(data=imp_12h.head(15), x='Importance', y='Feature', ax=axes[1])
axes[1].set_title('Top 15 Features (12-Hour Forecast)')

plt.tight_layout()
plt.show()


### 10. Visualization: Actual vs Predicted Scatter
Mapping the correlation of the model's accuracy.


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(physical_3h['target_water_level_3h'], physical_3h['pred'], alpha=0.3, color='blue', label='3-Hour Forecast')
plt.plot([ physical_3h['target_water_level_3h'].min(), physical_3h['target_water_level_3h'].max() ],
         [ physical_3h['target_water_level_3h'].min(), physical_3h['target_water_level_3h'].max() ], color='red', linestyle='--', lw=2, label='Perfect Prediction')

plt.title('Actual vs Predicted Water Levels (3H Horizon)')
plt.xlabel('Actual Water Level')
plt.ylabel('Predicted Water Level')
plt.legend()
plt.grid(True)
plt.show()


### 11. Visualization: Time-Series Overlay (Ellagawa)
Visually tracking the hardest river station across the test set.


In [ ]:
station_name = 'Ellagawa'
station_data = df_test_3h[df_test_3h['station'] == station_name].sort_values('datetime')

if not station_data.empty:
    plt.figure(figsize=(15, 6))
    plt.plot(station_data['datetime'], station_data['target_water_level_3h'], label='Actual (T+3h)', color='black', alpha=0.9)
    plt.plot(station_data['datetime'], station_data['pred'], label='Predicted (T+3h)', color='red', alpha=0.7, linestyle='--')
    
    plt.title(f'Real Timeline Simulation: Actual vs Predicted Water Levels - {station_name}')
    plt.xlabel('Date')
    plt.ylabel('Water Level (m)')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print(f"Station {station_name} not found in test set.")


### 12. Export Parallel Models


In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(model_3h, 'models/waterlevel_xgb_3h.joblib')
joblib.dump(model_12h, 'models/waterlevel_xgb_12h.joblib')
print("Successfully saved both 3H and 12H isolated models to /models/!")
